# WebScraping API da Betfair

### Instalando a biblioteca betfairlightweight

In [8]:
!pip install betfairlightweight

### Importando as Bibliotecas

In [9]:
from futpythontrader import *

import datetime
import pandas as pd
import pytz
import betfairlightweight

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

### Fazendo login na API da Betfair

In [10]:
from futpythontrader import *

# config = configparser.ConfigParser()
# config.sections()

username = "claudiamelotrader@gmail.com"
pw = "@L34ndr0F1lh0"
app_key = "UOjZNOLg4g2O6Ysl"

trading = betfairlightweight.APIClient(username, pw, app_key=app_key, cert_files=('./Betfair/MyAppBetfair.crt','./Betfair/MyAppBetfair.pem'))
trading.login()

APIError: None 
Params: None 
Exception: Could not find the TLS certificate file, invalid path: ./Betfair/MyAppBetfair.crt

In [ ]:
tt

### Criando o Filtro de Mercado

In [ ]:
dias = 1

filtros_mercado = betfairlightweight.filters.market_filter(
    event_type_ids=['1'], 
    market_start_time={
        'to': (datetime.datetime.utcnow() + datetime.timedelta(days=dias)).strftime("%Y-%m-%dT%TZ")
    }
)

### Listando Eventos e Competições

In [ ]:
eventos_futebol = trading.betting.list_events(filter=filtros_mercado)
competicoes_futebol = trading.betting.list_competitions(filter=filtros_mercado)

### Criando um Dicionário para Competições

In [ ]:
competitions = {event_object.competition.name: event_object.competition.id 
                for event_object in competicoes_futebol}

### Criando e Organizando o DataFrame

In [ ]:
df = pd.DataFrame(list(competitions.items()), columns=['League', 'Id']).sort_values('League').reset_index(drop=True)
df

### Obter IDs das Competições do DataFrame

In [ ]:
competition_ids = df['Id'].to_list()
competition_ids

### Criando Listas Vazias para Armazenar Informações

In [ ]:
eventos_list = []
mercado_evento_id = []
nomes_competicoes = []

In [ ]:
competition_ids = ['59','10932509'] 

### Processar Cada Competição

In [ ]:
for id_competicao in competition_ids:
    try:
        competicoes = trading.betting.list_competitions(
            filter=betfairlightweight.filters.market_filter(competition_ids=[id_competicao])
        )
        nome_competicao = competicoes[0].competition.name if competicoes else 'Competição não encontrada'
        nomes_competicoes.append(nome_competicao)
    except Exception as e:
        print(f"Erro ao processar competição {id_competicao}: {e}")
        continue

### Configurando Datas e Horários

In [ ]:
fuso_horario = 'America/Sao_Paulo' 
agora = datetime.datetime.now(pytz.timezone(fuso_horario)) 
data_atual = agora.date() 
data_futura = data_atual + datetime.timedelta(days=dias)

inicio_do_dia = datetime.datetime.combine(data_atual, datetime.time.min).astimezone(pytz.timezone(fuso_horario))
fim_do_dia = datetime.datetime.combine(data_futura, datetime.time.max).astimezone(pytz.timezone(fuso_horario))

### Definindo os Mercados Desejados

In [ ]:
mercados_desejados = [
    "Match Odds", "First Half Goals 0.5", "First Half Goals 1.5", "First Half Goals 2.5",
    "Over/Under 0.5 Goals", "Over/Under 1.5 Goals", "Over/Under 2.5 Goals", "Over/Under 3.5 Goals",
    "Over/Under 4.5 Goals", "Over/Under 5.5 Goals", "Over/Under 6.5 Goals", "Over/Under 7.5 Goals",
    "Over/Under 8.5 Goals", "Both teams to Score?", "Correct Score",
    ]

### Configurando o Filtro de Preços

In [ ]:
price_filter = betfairlightweight.filters.price_projection(
    price_data=['EX_BEST_OFFERS'],
    ex_best_offers_overrides=betfairlightweight.filters.ex_best_offers_overrides(best_prices_depth=1)
)

### Iteração com a API da Betfair

In [ ]:
# Processando Cada Competição
for competition_id in competition_ids:
    nome_competicao = nomes_competicoes[competition_ids.index(competition_id)]
    
    filtros_mercado = betfairlightweight.filters.market_filter(
        event_type_ids=['1'], 
        market_start_time={'from': inicio_do_dia.strftime('%Y-%m-%dT%H:%M:%SZ'), 'to': fim_do_dia.strftime('%Y-%m-%dT%H:%M:%SZ')},
        competition_ids=[competition_id]
    )
    
    # Obtendo e Filtrando Eventos por Competição
    try:
        eventos_futebol = trading.betting.list_events(filter=filtros_mercado)
    except Exception as e:
        print(f"Erro ao listar eventos para competição {competition_id}: {e}")
        continue
    
    # Processando Cada Evento
    for obj_evento in eventos_futebol:
        id_evento = obj_evento.event.id

        filtro_catalogo_mercados = betfairlightweight.filters.market_filter(event_ids=[id_evento])
        
        # Filtrando e Obtendo Catálogos de Mercado
        try:
            catalogos_mercado = trading.betting.list_market_catalogue(
                filter=filtro_catalogo_mercados, max_results='100', sort='FIRST_TO_START', market_projection=['RUNNER_METADATA']
            )
        except Exception as e:
            print(f"Erro ao listar catálogo de mercado para evento {id_evento}: {e}")
            continue

        market_ids = {}
        odds_dict = {}
        selection_ids = {}
        
        # Processando Cada Mercado no Catálogo
        for mercado in catalogos_mercado:
            if mercado.market_name in mercados_desejados:
                market_ids['IDMercado_' + mercado.market_name.replace(" ", "_").replace(".", "")] = mercado.market_id

                try:
                    market_books = trading.betting.list_market_book(market_ids=[mercado.market_id], price_projection=price_filter)
                except Exception as e:
                    print(f"Erro ao listar book de mercado {mercado.market_id}: {e}")
                    continue
                
                # Obtendo e Processando Informações do Mercado
                for idx, runner in enumerate(market_books[0].runners):
                    try:
                        back_price = runner.ex.available_to_back[0].price if runner.ex.available_to_back else 1.01
                        lay_price = runner.ex.available_to_lay[0].price if runner.ex.available_to_lay else 1000
                        
                        if mercado.market_name == "Match Odds":
                            selection_key = None
                            if idx == 0:
                                nome_coluna = "Match_Odds_Home"
                                selection_key = 'ID_Home'
                            elif idx == 1:
                                nome_coluna = "Match_Odds_Away"
                                selection_key = 'ID_Away'
                            elif idx == 2:
                                nome_coluna = "Match_Odds_Draw"
                                selection_key = 'ID_Draw'
                            if selection_key:
                                selection_ids[selection_key] = runner.selection_id
                        else:
                            nome_coluna = f"{mercado.market_name.replace(' ', '_')}_{runner.selection_id}"

                        odds_dict[nome_coluna] = {'Back': back_price, 'Lay': lay_price}
                    except IndexError:
                        odds_dict[nome_coluna] = {'Back': 1.01, 'Lay': 1000}

        nome_evento = obj_evento.event.name
        times = nome_evento.split(" v ")
        if len(times) != 2:
            print(f"Formato inesperado do nome do evento: {nome_evento}")
            continue

        data_hora = obj_evento.event.open_date.replace(tzinfo=datetime.timezone.utc).astimezone(tz=None)
        
        # Criando e Atualizando um Dicionário de Eventos
        evento_dict = {
            'ID_Evento': id_evento,
            'League': nome_competicao,
            'Date': data_hora.strftime('%Y-%m-%d'),
            'Time': data_hora.strftime('%H:%M:%S'),
            'Home': times[0],
            'Away': times[1]
        }

        evento_dict.update(odds_dict)
        evento_dict.update(market_ids)
        evento_dict.update(selection_ids)
        eventos_list.append(evento_dict)

# Criando uma DataFrame com os Eventos
planilha_eventos_futebol = pd.DataFrame(eventos_list)

In [ ]:
planilha_eventos_futebol

### Selecionando Colunas do DataFrame

In [ ]:
df = planilha_eventos_futebol[['League','Date','Time','Home','Away',
'Match_Odds_Home','Match_Odds_Away','Match_Odds_Draw',
'First_Half_Goals_0.5_5851482','First_Half_Goals_0.5_5851483',
'First_Half_Goals_1.5_1221385','First_Half_Goals_1.5_1221386',
'First_Half_Goals_2.5_47972','First_Half_Goals_2.5_47973',
'Over/Under_0.5_Goals_5851482','Over/Under_0.5_Goals_5851483',
'Over/Under_1.5_Goals_1221385','Over/Under_1.5_Goals_1221386',
'Over/Under_2.5_Goals_47972','Over/Under_2.5_Goals_47973',
'Over/Under_3.5_Goals_1222344','Over/Under_3.5_Goals_1222345',
'Over/Under_4.5_Goals_1222347','Over/Under_4.5_Goals_1222346',
'Over/Under_5.5_Goals_1485567','Over/Under_5.5_Goals_1485568',
'Over/Under_6.5_Goals_2542448','Over/Under_6.5_Goals_2542449',
'Over/Under_7.5_Goals_1485572','Over/Under_7.5_Goals_1485573',
'Over/Under_8.5_Goals_2407528','Over/Under_8.5_Goals_2407529',
'Both_teams_to_Score?_30246','Both_teams_to_Score?_110503',
'Correct_Score_1','Correct_Score_4','Correct_Score_9','Correct_Score_16',
'Correct_Score_2','Correct_Score_3','Correct_Score_8','Correct_Score_15',
'Correct_Score_5','Correct_Score_6','Correct_Score_7','Correct_Score_14',
'Correct_Score_10','Correct_Score_11','Correct_Score_12','Correct_Score_13',
'Correct_Score_9063254','Correct_Score_9063255','Correct_Score_9063256']]

### Renomeando Colunas

In [ ]:
df.columns = ['League','Date','Time','Home','Away',
'Odd_H','Odd_A','Odd_D',
'Odd_Under05_HT','Odd_Over05_HT',
'Odd_Under15_HT','Odd_Over15_HT',
'Odd_Under25_HT','Odd_Over25_HT',
'Odd_Under05_FT','Odd_Over05_FT',
'Odd_Under15_FT','Odd_Over15_FT',
'Odd_Under25_FT','Odd_Over25_FT',
'Odd_Under35_FT','Odd_Over35_FT',
'Odd_Under45_FT','Odd_Over45_FT',
'Odd_Under55_FT','Odd_Over55_FT',
'Odd_Under65_FT','Odd_Over65_FT',
'Odd_Under75_FT','Odd_Over75_FT',
'Odd_Under85_FT','Odd_Over85_FT',
'Odd_BTTS_Yes','Odd_BTTS_No',
'Odd_CS_0x0','Odd_CS_0x1','Odd_CS_0x2','Odd_CS_0x3',
'Odd_CS_1x0','Odd_CS_1x1','Odd_CS_1x2','Odd_CS_1x3',
'Odd_CS_2x0','Odd_CS_2x1','Odd_CS_2x2','Odd_CS_2x3',
'Odd_CS_3x0','Odd_CS_3x1','Odd_CS_3x2','Odd_CS_3x3',
'Odd_CS_Goleada_H','Odd_CS_Goleada_A','Odd_CS_Goleada_D']

### Preparando uma Lista de Colunas Específicas

In [ ]:
colunas_para_separar = [
    'Odd_H', 'Odd_A', 'Odd_D', 'Odd_Under05_HT', 'Odd_Over05_HT', 'Odd_Under15_HT', 'Odd_Over15_HT', 
    'Odd_Under25_HT', 'Odd_Over25_HT', 'Odd_Under05_FT', 'Odd_Over05_FT', 'Odd_Under15_FT', 'Odd_Over15_FT', 
    'Odd_Under25_FT', 'Odd_Over25_FT', 'Odd_Under35_FT', 'Odd_Over35_FT', 'Odd_Under45_FT', 'Odd_Over45_FT', 
    'Odd_Under55_FT', 'Odd_Over55_FT', 'Odd_Under65_FT', 'Odd_Over65_FT', 'Odd_Under75_FT', 'Odd_Over75_FT', 
    'Odd_Under85_FT', 'Odd_Over85_FT', 'Odd_BTTS_Yes', 'Odd_BTTS_No', 'Odd_CS_0x0', 'Odd_CS_0x1', 
    'Odd_CS_0x2', 'Odd_CS_0x3', 'Odd_CS_1x0', 'Odd_CS_1x1', 'Odd_CS_1x2', 'Odd_CS_1x3', 'Odd_CS_2x0', 
    'Odd_CS_2x1', 'Odd_CS_2x2', 'Odd_CS_2x3', 'Odd_CS_3x0', 'Odd_CS_3x1', 'Odd_CS_3x2', 'Odd_CS_3x3', 
    'Odd_CS_Goleada_H', 'Odd_CS_Goleada_A', 'Odd_CS_Goleada_D'
]

### Criando um Novo DataFrame

In [ ]:
jogos_do_dia = pd.DataFrame()

### Iterando Pelas Colunas do DataFrame Original e Processando Colunas Específicas

In [ ]:
for coluna in df.columns:
    if coluna in colunas_para_separar:
        jogos_do_dia[coluna + '_Back'] = df[coluna].apply(lambda x: x['Back'] if isinstance(x, dict) and 'Back' in x else None)
        jogos_do_dia[coluna + '_Lay'] = df[coluna].apply(lambda x: x['Lay'] if isinstance(x, dict) and 'Lay' in x else None)
    else:
        jogos_do_dia[coluna] = df[coluna]

### Processamento do DataFrame para Hoje e Amanhã

In [ ]:
def hoje():
    from datetime import date, datetime, timedelta
    dia = date.today() + timedelta(0)
    return dia

def amanha():
    from datetime import date, datetime, timedelta
    dia = date.today() + timedelta(1)
    return dia

def depois_de_amanha():
    from datetime import date, datetime, timedelta
    dia = date.today() + timedelta(2)
    return dia

def drop_reset_index(df):
    df = df.dropna()
    df = df.reset_index(drop=True)
    df.index += 1
    return df

hoje = str(hoje())

flt = jogos_do_dia.Date == hoje
df_hoje = jogos_do_dia[flt]
df_hoje = df_hoje.sort_values(['Time'])
df_hoje = drop_reset_index(df_hoje)
display(df_hoje)

amanha = str(amanha())

flt = jogos_do_dia.Date == amanha
df_amanha = jogos_do_dia[flt]
df_amanha = df_amanha.sort_values(['Time'])
df_amanha = drop_reset_index(df_amanha)
display(df_amanha)
# df_amanha.to_excel(f"Jogos_do_Dia_Betfair_Back_Lay_{amanha}.xlsx",index=False)

depois_de_amanha = str(depois_de_amanha())

flt = jogos_do_dia.Date == depois_de_amanha
df_depois_amanha = jogos_do_dia[flt]
df_depois_amanha = df_depois_amanha.sort_values(['Time'])
df_depois_depois_amanha = drop_reset_index(df_depois_amanha)
display(df_depois_amanha)
# df_amanha.to_excel(f"Jogos_do_Dia_Betfair_Back_Lay_{amanha}.xlsx",index=False)

print("Finalizado!")